# Chapter 3 — Matching the Dispersion Suppressor to the Straight Section

## Goal

We build the Chapter 3 matching section from the optimized forward dispersion suppressor produced in the previous chapter. Start with a copy of that result, then add four new matching quadrupoles:

```text
QFF2, QDF2, QFF3, QDF3
```

Their strengths are optimized so that the Twiss parameters at the end of the new line match the periodic Twiss parameters of the straight section ahead in the beam direction:

$$
(\beta_a,\alpha_a,\beta_b,\alpha_b)_\mathrm{END}
=
(\beta_a,\alpha_a,\beta_b,\alpha_b)_\mathrm{SS}.
$$


## Workflow

```text
copy the Chapter 2 forward suppressor
→ add DB
→ add QFF2, QDF2, QFF3, QDF3
→ build MSSF
→ build ARC_TO_SSF = DISPSUPF + MSSF
→ compute the Twiss residuals
→ compute the residual Jacobian with GTPSA
→ optimize the four quadrupole strengths
→ save the optimized strengths
```


## Packages


In [ ]:
using SciBmad
using GTPSA
using DifferentiationInterface
import DifferentiationInterface as DI

using LinearAlgebra
using Printf


## 1. Copy the Chapter 2 forward suppressor

This notebook assumes that the previous chapter has produced an optimized forward dispersion suppressor beamline object called `dispsupF_ch2`.

We make a working copy for this chapter:


In [ ]:
if @isdefined(dispsupF_ch2)
    dispsupF = deepcopy(dispsupF_ch2)
    println("Copied dispsupF_ch2 → dispsupF")
else
    @warn """
    dispsupF_ch2 is not defined.

    Before running the full notebook, load the Chapter 2 result so that

        dispsupF_ch2

    is the optimized forward dispersion suppressor beamline.
    """
end


## 2. Define the straight-section drift

The matching section uses the straight-section drift

```text
DB: Drift, L = 5.855
```

We also reuse the arc drift lengths `D1` and `D2`.


In [ ]:
D1_len = 0.609
D2_len = 1.241
DB_len = 5.855
L_quad = 0.5

D1() = Drift(L=D1_len)
D2() = Drift(L=D2_len)
DB() = Drift(L=DB_len)


## 3. Define the four matching quadrupoles

The four new quadrupoles are

```text
QFF2, QDF2, QFF3, QDF3
```

The initial strengths are chosen from the straight-section FODO quadrupole strength.


In [ ]:
K_ss = 0.351957452649287

K_start = [
     K_ss,   # QFF2
    -K_ss,   # QDF2
     K_ss,   # QFF3
    -K_ss,   # QDF3
]

QFF2(k) = Quadrupole(L=L_quad, Kn1=k)
QDF2(k) = Quadrupole(L=L_quad, Kn1=k)
QFF3(k) = Quadrupole(L=L_quad, Kn1=k)
QDF3(k) = Quadrupole(L=L_quad, Kn1=k)

println("Initial matching quadrupole strengths:")
@printf("QFF2 Kn1 = %.15f\n", K_start[1])
@printf("QDF2 Kn1 = %.15f\n", K_start[2])
@printf("QFF3 Kn1 = %.15f\n", K_start[3])
@printf("QDF3 Kn1 = %.15f\n", K_start[4])


## 4. Build the matching section `MSSF`

The forward matching section is

```text
QFF2, D1, DB, D2,
QDF2, D1, DB, D2,
QFF3, D1, DB, D2,
QDF3, D1, DB, D2
```

We write it as a function of the four quadrupole strengths.


In [ ]:
function build_MSSF(k; species_ref=Species("electron"), E_ref=18e9)
    elements = [
        QFF2(k[1]), D1(), DB(), D2(),
        QDF2(k[2]), D1(), DB(), D2(),
        QFF3(k[3]), D1(), DB(), D2(),
        QDF3(k[4]), D1(), DB(), D2(),
    ]

    return Beamline(elements; species_ref=species_ref, E_ref=E_ref)
end

mssf0 = build_MSSF(K_start)


## 5. Build `ARC_TO_SSF`

The complete line is

```text
ARC_TO_SSF = DISPSUPF + MSSF
```

For the optimization, the varying part is `MSSF`. The input Twiss parameters at the entrance of `MSSF` are the Twiss parameters at the end of `DISPSUPF`.


In [ ]:
function build_ARC_TO_SSF(k)
    mssf = build_MSSF(k)

    if @isdefined(dispsupF)
        # Depending on the exact Beamline representation in your SciBmad version,
        # replace this line by the corresponding beamline concatenation method if needed.
        return Beamline([dispsupF, mssf]; species_ref=mssf.species_ref, E_ref=18e9)
    else
        @warn "dispsupF is not defined, returning only MSSF."
        return mssf
    end
end


## 6. Set the input and target Twiss parameters

The target values are the periodic Twiss parameters of the forward straight-section FODO cell.

The input values should come from the end of `DISPSUPF`. Replace the placeholder input values with the Chapter 2 output before running a full match.


In [ ]:
struct Twiss
    beta::Float64
    alpha::Float64
end

target_a = Twiss(27.20598820, -2.40249037)
target_b = Twiss(4.96091411,   0.48460556)

if @isdefined(twiss_after_dispsupF)
    input_a = Twiss(twiss_after_dispsupF.beta_a,  twiss_after_dispsupF.alpha_a)
    input_b = Twiss(twiss_after_dispsupF.beta_b,  twiss_after_dispsupF.alpha_b)
else
    @warn """
    twiss_after_dispsupF is not defined.

    Define it from the Chapter 2 result before running the optimization:

        twiss_after_dispsupF = (
            beta_a  = ...,
            alpha_a = ...,
            beta_b  = ...,
            alpha_b = ...
        )
    """
    input_a = Twiss(30.0, -2.0)
    input_b = Twiss(5.0,   0.5)
end

println("Input Twiss at the entrance of MSSF:")
@printf("beta.a  = %.10f, alpha.a = %.10f\n", input_a.beta, input_a.alpha)
@printf("beta.b  = %.10f, alpha.b = %.10f\n", input_b.beta, input_b.alpha)

println("\nTarget Twiss at END:")
@printf("beta.a  = %.10f, alpha.a = %.10f\n", target_a.beta, target_a.alpha)
@printf("beta.b  = %.10f, alpha.b = %.10f\n", target_b.beta, target_b.alpha)


## 7. Track one particle through a beamline

This function returns the exit coordinates of one particle. GTPSA will be used through `DifferentiationInterface` to differentiate this tracking result.


In [ ]:
function track_a_particle(v0, beamline)
    v = similar(v0)
    v .= v0

    b = Bunch(v; species=beamline.species_ref, p_over_q_ref=beamline.p_over_q_ref)
    track!(b, beamline)

    return b.coords.v
end


## 8. Compute the transfer matrix with GTPSA

The transfer matrix through `MSSF` is the Jacobian of the final phase-space coordinates with respect to the initial phase-space coordinates.

We compute that Jacobian using `AutoGTPSA()`.


In [ ]:
function transfer_matrix_gtpsa(beamline; x0=zeros(6))
    prep = DI.prepare_jacobian(
        track_a_particle,
        AutoGTPSA(),
        x0,
        Constant(beamline),
    )

    return DI.jacobian(
        track_a_particle,
        prep,
        AutoGTPSA(),
        x0,
        Constant(beamline),
    )
end

function transverse_blocks(J)
    Mx = J[1:2, 1:2]
    My = J[3:4, 3:4]
    return Mx, My
end


## 9. Propagate Twiss parameters

Given a transverse transfer matrix `M`, Twiss parameters propagate through

$$
\Sigma_2 = M \Sigma_1 M^T,
$$

where

$$
\Sigma =
\begin{pmatrix}
\beta & -\alpha \\
-\alpha & \gamma
\end{pmatrix},
\qquad
\gamma = \frac{1+\alpha^2}{\beta}.
$$


In [ ]:
gamma(t::Twiss) = (1 + t.alpha^2) / t.beta

function sigma_matrix(t::Twiss)
    return [
         t.beta    -t.alpha
        -t.alpha    gamma(t)
    ]
end

function twiss_from_sigma(S)
    return Twiss(S[1, 1], -S[1, 2])
end

function propagate_twiss(t::Twiss, M)
    S2 = M * sigma_matrix(t) * transpose(M)
    return twiss_from_sigma(S2)
end

function end_twiss(k)
    mssf = build_MSSF(k)
    J = transfer_matrix_gtpsa(mssf)
    Mx, My = transverse_blocks(J)

    end_a = propagate_twiss(input_a, Mx)
    end_b = propagate_twiss(input_b, My)

    return end_a, end_b
end


## 10. Define the matching residual

The matching residual has four components:

```text
beta.a  at END  - beta.a target
alpha.a at END  - alpha.a target
beta.b  at END  - beta.b target
alpha.b at END  - alpha.b target
```

The beta residuals are normalized by the target beta values so that beta and alpha residuals have comparable scale.


In [ ]:
function matching_residual(k)
    end_a, end_b = end_twiss(k)

    return [
        (end_a.beta  - target_a.beta) / target_a.beta,
         end_a.alpha - target_a.alpha,
        (end_b.beta  - target_b.beta) / target_b.beta,
         end_b.alpha - target_b.alpha,
    ]
end

function merit(k)
    r = matching_residual(k)
    return sum(abs2, r)
end

println("Initial residual:")
println(matching_residual(K_start))
println("Initial merit = ", merit(K_start))


## 11. Compute the residual Jacobian with GTPSA

The optimizer needs

$$
J_{ij} =
\frac{\partial r_i}{\partial K_j},
$$

where

```text
r_i = one of the four Twiss residuals
K_j = one of QFF2, QDF2, QFF3, QDF3
```

We compute this Jacobian using `AutoGTPSA()`.


In [ ]:
function residual_jacobian_gtpsa(k)
    prep = DI.prepare_jacobian(
        matching_residual,
        AutoGTPSA(),
        k,
    )

    return DI.jacobian(
        matching_residual,
        prep,
        AutoGTPSA(),
        k,
    )
end

J0 = residual_jacobian_gtpsa(K_start)

println("Initial residual Jacobian:")
display(J0)


## 12. Optimize the four strengths

The matching loop uses the GTPSA residual Jacobian at each step. Since there are four residuals and four variables, a damped Gauss-Newton step is a natural choice:

$$
(J^T J + \lambda I)\Delta K = -J^T r.
$$

Then update

$$
K \leftarrow K + \Delta K.
$$


In [ ]:
function match_with_gtpsa_jacobian(k0; maxiter=40, tol=1e-12, λ0=1e-3)
    k = copy(k0)
    λ = λ0
    history = NamedTuple[]

    for iter in 1:maxiter
        r = matching_residual(k)
        J = residual_jacobian_gtpsa(k)
        merit_now = sum(abs2, r)

        A = transpose(J) * J + λ * I
        step = -(A \ (transpose(J) * r))

        trial = k + step
        merit_trial = merit(trial)

        if merit_trial < merit_now
            k = trial
            λ /= 10
        else
            λ *= 10
        end

        push!(history, (
            iter = iter,
            merit = merit_now,
            step_norm = norm(step),
            lambda = λ,
        ))

        if merit_now < tol || norm(step) < tol
            break
        end
    end

    return k, history
end

K_match, history = match_with_gtpsa_jacobian(K_start)

println("Optimization history:")
for h in history
    @printf("%3d  merit = %.6e  step = %.3e  lambda = %.3e\n",
            h.iter, h.merit, h.step_norm, h.lambda)
end

println("\nOptimized strengths:")
@printf("QFF2 Kn1 = %.15f\n", K_match[1])
@printf("QDF2 Kn1 = %.15f\n", K_match[2])
@printf("QFF3 Kn1 = %.15f\n", K_match[3])
@printf("QDF3 Kn1 = %.15f\n", K_match[4])

println("\nFinal residual:")
println(matching_residual(K_match))
println("Final merit = ", merit(K_match))


## 13. Check the matched Twiss parameters


In [ ]:
matched_a, matched_b = end_twiss(K_match)

@printf("%-10s %18s %18s %18s\n", "quantity", "target", "matched", "difference")
@printf("%-10s %18.10e %18.10e %18.10e\n", "beta.a",  target_a.beta,  matched_a.beta,  matched_a.beta  - target_a.beta)
@printf("%-10s %18.10e %18.10e %18.10e\n", "alpha.a", target_a.alpha, matched_a.alpha, matched_a.alpha - target_a.alpha)
@printf("%-10s %18.10e %18.10e %18.10e\n", "beta.b",  target_b.beta,  matched_b.beta,  matched_b.beta  - target_b.beta)
@printf("%-10s %18.10e %18.10e %18.10e\n", "alpha.b", target_b.alpha, matched_b.alpha, matched_b.alpha - target_b.alpha)


## 14. Save the optimized strengths

Save the optimized quadrupole strengths in a Julia file so they can be loaded by later chapters.


In [ ]:
open("mSSF_strengths.jl", "w") do io
    println(io, "# Chapter 3 optimized forward matching strengths")
    @printf(io, "K_QFF2 = %.17g\n", K_match[1])
    @printf(io, "K_QDF2 = %.17g\n", K_match[2])
    @printf(io, "K_QFF3 = %.17g\n", K_match[3])
    @printf(io, "K_QDF3 = %.17g\n", K_match[4])
end

println("Wrote mSSF_strengths.jl")


## Exercises

### Reverse matching section

Construct the reverse matching section using the same procedure. Reverse the drift ordering and optimize the four reverse matching quadrupoles:

```text
QFR2, QDR2, QFR3, QDR3
```

Save the optimized strengths as

```text
mSSR_strengths.jl
```

Compare the reverse matching strengths with the forward matching strengths.

### Merit weight

Restart from the initial strengths and modify the merit function by putting a large weight on the first beta residual. For example:

```julia
weights = [1e10, 1, 1, 1]
```

Then use

```julia
sum(weights .* abs2.(matching_residual(k)))
```

in the matching loop and compare the final residuals with the equal-weight case.
